In [8]:
from openai import OpenAI
from dotenv import load_dotenv

import os
import json
from IPython.display import Markdown

load_dotenv(override=True)

True

In [9]:
llm_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPEN_ROUTER_KEY")
)

MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free, NVIDIA)

In [10]:
menu = [
    {"name": "Zinger Burger", "price": 850},
    {"name": "Chicken Biryani", "price": 450},
    {"name": "Club Sandwich", "price": 700}
]

complaint_list = []
orders = [
    {"order_id": "OR-1777"},
    {"order_id": "OR-1234"},
    {"order_id": "OR-0001"}
]

In [11]:
def fetch_menu():
    return menu

def log_complaint(customer_name, order_id, phone, complaint):
    temp_dict = {
        "customer_name": customer_name,
        "order_id": order_id,
        "phone": phone,
        "complaint": complaint,
        "status": "Pending"
    }
    complaint_list.append(temp_dict)

    return {
        "message": f"Saved! Total Complaint: {len(complaint_list)}"
    }
def validate_order_id(order_id):
    for order in orders:
        if order["order_id"] == order_id:
            return {
                "valid": True,
                "message": "Valid order ID"
            }

    return {
        "valid": False,
        "message": "Invalid order ID"
    }


tools = [
    {
        "type": "function",
        "function": {
                "name": "fetch_menu",
                "description": "Get the restaurant menu",
                # no inputs
                "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "log_complaint",
            "description": "Log the complain",
            "strict": True,
            "parameters": {
                "type": "object", 
                "properties": {
                    "customer_name": {
                        "type": "string",
                        "description": "Customer name who is logging complain.",
                    },
                    "order_id": {
                        "type": "string",
                        "description": "the order id of which customer ordered food. OrderId should be like this OR-0000",
                    },
                    "phone": {
                        "type": "string",
                        "description": "Customer phone number or '-' ",
                    },
                    "complaint": {
                        "type": "string",
                        "description": "customer complain should me not more than 8 words",
                    }
                },
                "required": ["customer_name", "order_id", "phone", "complaint"],
                "additionalProperties": False,

            },
        }
    },
    {
    "type": "function",
    "function": {
        "name": "validate_order_id",
        "description": "Validate whether the given order ID exists.",
        "strict": True,
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order ID to validate, such as OR-0000."
                }
            },
            "required": ["order_id"],
            "additionalProperties": False
        }
    }
}

]

In [12]:
messages = [
    {
      "role": "system",  # User | Assistant | System
      "content": """
       You are the AI Customer Support Assistant for **Xander Restaurant**. Your job is to answer customer queries in a friendly, professional, and helpful way.

        ### Tools
        1. **`fetch_menu`** — Call whenever customers ask about menu items, prices, ingredients, or food recommendations. Use the tool's response to provide accurate information.
        2. **`log_complaint`** — Call whenever a customer raises a complaint. Collect the necessary details and log the complaint using this schema:
        3. **`validate_order_id`** — Call whenever a customer provides an order ID that needs validation. Pass the order ID to the tool and use its response to determine whether the order ID is valid before proceeding.

        ```python
        complaint = {
            "customer_name": "",
            "phone": "",
            "complaint": "",
            "order_id": "",
            "status": "Pending"
        }
        ```

        Ask for missing necessary details before logging. The `order_id` can be empty if unavailable. Set the initial status to `"Pending"`. Confirm the complaint is logged only after the tool succeeds.

        ### Guidelines
        - Keep replies short, natural, and friendly.
        - Never invent menu details, prices, or restaurant policies.
        - If you don't know something, politely say so.
        - Make every customer feel valued.
        - All prices are in Pakistani Rupees. Write them as "Rs 850".
      """
    }
]


In [13]:
function_names = {
    "fetch_menu": fetch_menu,
    "log_complaint": log_complaint,
    "validate_order_id":validate_order_id
}


def ask_assistant(question):

    messages.append({
        "role": "user",
        "content": question
    })

    completion = llm_client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    reply = completion.choices[0].message

    if reply.tool_calls:
        messages.append(reply)

        for call in reply.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            function_name = call.function.name

            # Validate the SAME order ID before logging a complaint
            if function_name == "log_complaint":
                order_id = args["order_id"]
                validation = validate_order_id(order_id)

                if not validation["valid"]:
                    result = {
                        "error": "Invalid order ID. Complaint was not saved."
                    }
                else:
                    result = log_complaint(**args)

            else:
                result = function_names[function_name](**args)

            print("Tool called:", function_name, args, result)

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result)
            })

        completion = llm_client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    final_reply = completion.choices[0].message.content

    print(final_reply)

    return Markdown(final_reply)


In [ ]:
ask_assistant("i ordered burger but roll was served WTH!!! and the bill was made with burger, my name is Sam, OR-1777, phone is 0322342423")
ask_assistant("I want to complain about my order OR-1234. My name is Ali, phone is 03001234567, and my food arrived cold.")
ask_assistant(
    "I want to complain about my order OR-234. "
    "My name is Ali, phone is 03001234567, "
    "and my food arrived cold."
)

Tool called: log_complaint {'customer_name': 'Ali', 'phone': '03001234567', 'order_id': 'OR-1234', 'complaint': 'food arrived cold'} {'message': 'Saved! Total Complaint: 3'}
Your complaint has been logged, Ali! 🙏 We’ve noted that the food arrived cold for order **OR‑1234**. Thanks for letting us know, and we’re sorry for the frustration. Is there anything else we can do for you?


Your complaint has been logged, Ali! 🙏 We’ve noted that the food arrived cold for order **OR‑1234**. Thanks for letting us know, and we’re sorry for the frustration. Is there anything else we can do for you?

In [15]:
print(json.dumps(complaint_list))

[{"customer_name": "Ali", "order_id": "OR-1234", "phone": "03001234567", "complaint": "food arrived cold", "status": "Pending"}]
